In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from pathlib import Path
from typing import List, Dict, Any, Optional
from tqdm.auto import tqdm
from pypdf import PdfReader
from dotenv import load_dotenv

# Langhchain framework
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# HF + Groq clients
from huggingface_hub import InferenceClient
from groq import Groq

import warnings
warnings.filterwarnings("ignore")

## Load API keys for credential key

In [2]:
ENV_PATH = Path.cwd().parent.parent / "credit_risk_production" / ".env"
load_dotenv(dotenv_path=ENV_PATH)

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(f"HF token loaded:  {HUGGINGFACE_API_KEY[:3]}...")
print(f"GROQ token loaded: {GROQ_API_KEY[:3]}...")

HF token loaded:  hf_...
GROQ token loaded: gsk...


## Load database

In [3]:
DATA = Path.cwd().parent.parent / "credit_risk_production" / "database" / "data" / "merged_credit_risk_data.parquet"
FEATURES = Path.cwd().parent.parent / "credit_risk_production" / "database" / "data" / "features_data.parquet"

df = pd.read_parquet(DATA)
features = pd.read_parquet(FEATURES)

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Features loaded: {features.shape[0]} rows, {features.shape[1]} columns")

Data loaded: 51336 rows, 87 columns
Features loaded: 51336 rows, 47 columns


## Load Fraud ML Models

In [4]:
# Load ML models from bundle
MODEL_BUNDLE = Path.cwd().parent.parent / "data_science" / "models" / "fraud_models" / "model_bundle.joblib"
with open(MODEL_BUNDLE, "rb") as f:
    model_bundle = joblib.load(f)
    print(f"Model bundle loaded: {list(model_bundle.keys())}")

# Load best parameters for each model
PARAMS = Path.cwd().parent.parent / "data_science" / "models" / "metrics" / "best_parameters.json"
with open(PARAMS, "r") as f:
    best_params = json.load(f)
    print(f"Best parameters loaded: {list(best_params.keys())}")

# Load metrics model
METRICS = Path.cwd().parent.parent / "data_science" / "models" / "metrics" / "model_metrics.csv"
metrics_df = pd.read_csv(METRICS)
display(metrics_df)

Model bundle loaded: ['models', 'scaler', 'label_encoders', 'feature_columns', 'class_labels']
Best parameters loaded: ['Logistic Regression', 'Random Forest', 'Decision Tree', 'XGBoost', 'K-Nearest Neighbors']


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.957148,0.955984,0.988137,0.971795,0.958286
1,Random Forest,0.982859,0.986372,0.990744,0.988554,0.998319
2,Decision Tree,0.986560,0.995266,0.986703,0.990966,0.993473
3,XGBoost,0.987924,0.998547,0.985269,0.991864,0.998564
4,K-Nearest Neighbors,0.869108,0.885370,0.947464,0.915365,0.906852


In [5]:
model_bundle['models']

{'Logistic Regression': LogisticRegression(C=10, random_state=42),
 'Random Forest': RandomForestClassifier(bootstrap=False, min_samples_leaf=2,
                        min_samples_split=10, random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=4, min_samples_split=5,
                        random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=1.0, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               gamma=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.01, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=5, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
        

In [6]:
best_params

{'Logistic Regression': {'solver': 'lbfgs', 'penalty': 'l2', 'C': 10},
 'Random Forest': {'n_estimators': 100,
  'min_samples_split': 10,
  'min_samples_leaf': 2,
  'max_depth': None,
  'bootstrap': False},
 'Decision Tree': {'min_samples_split': 5,
  'min_samples_leaf': 4,
  'max_depth': 10},
 'XGBoost': {'n_estimators': 200,
  'max_depth': 5,
  'learning_rate': 0.01,
  'colsample_bytree': 1.0},
 'K-Nearest Neighbors': {'weights': 'distance',
  'n_neighbors': 9,
  'metric': 'manhattan'}}

# 📚 Document Ingestion — PDF → Chunks (per-document strategy)

In [7]:
PDF_DIR = Path("../../credit_risk_production/database/pdf")

pdf_files = {
    "delinquency":  PDF_DIR / "Delinquency_Classification .pdf",
    "fraud":        PDF_DIR / "Fraud_Typologies_and_Red Flags .pdf",
    "regulatory":   PDF_DIR / "Regulatory_Risk Policy Core .pdf",
    "scorecard":    PDF_DIR / "Scorecard_Cut-off Policy.pdf"
}

raw_texts: Dict[str, str] = {}

for name, path in pdf_files.items():
    reader = PdfReader(str(path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    raw_texts[name] = text
    print(f"{name:12s} | pages={len(reader.pages):3d} | chars={len(text):,}")

delinquency  | pages= 93 | chars=191,238
fraud        | pages=  9 | chars=25,113
regulatory   | pages= 11 | chars=34,419
scorecard    | pages=178 | chars=465,508


## Chunking strategy to retrieve Document augmentation for each PDFs to enrich vocab on LLM
- #### Strategy 1: Delinquency Classification — rule-based chunking
- #### Strategy 2: Fraud Typologies — focusing on fraud paragraph to be chunked
- #### Strategy 3: Regulatory Policy — recursive with heading preservation
- #### Strategy 4: Scorecard Cut-off — table-aware chunking
- #### Final Strategy: Combine all chunks

In [8]:
def chunk_delinquency(text: str) -> List[Document]:
    """Split by classification headings, falls back to paragraph split."""
    keywords = ["Standard", "Sub-standard", "Substandrad", "Doubtful", "Loss"]

    # Crude split: find the index of each keyword and slice
    positions = []
    for kw in keywords:
        idx = text.lower().find(kw.lower())
        if idx != -1:
            positions.append((idx, kw))
    positions.sort()

    docs: List[Document] = []
    if not positions:

        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "delinquency", "chunk_id": i, "class_level": "unknown"}
            ))
        return docs

    for i, (start, kw) in enumerate(positions):
        end = positions[i + 1][0] if i + 1 < len(positions) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue

        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "delinquency", "class_level": kw.lower(), "chunk_id": i},
        ))

    return docs

# Usage on delinquency PDF
deling_docs = chunk_delinquency(raw_texts["delinquency"])
print(f"Delinquency chunks: {len(deling_docs)}")
print(deling_docs[3].page_content[:300], "\n--")

Delinquency chunks: 4
sub-standard' immediately on restructuring, 
all borrowers, with the exception of the borrowal categories specified in para 14.1 below ( i.e 
consumer and personal advances, advances classified as capital market and real estate 
exposures), will be entitled to retain the asset classification upon re 
--


#### Strategy 2: Fraud Typologies — focusing on fraud paragraph to be chunked

In [13]:
def chunk_fraud(text: str) -> List[Document]:
    """Split by typology headings, falls back to paragraph split."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=80,
        separators=["\n\n\n", "\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "fraud", "chunk_id": i, "typology": "unspecified"}
        ))
    return docs

# Usage on fraud PDF
fraud_docs = chunk_fraud(raw_texts["fraud"])
print(f"Fraud chunks: {len(fraud_docs)}")
print(fraud_docs[3].page_content[:300], "\n--")

Fraud chunks: 36
platforms achieved a 61% improvement in detection accuracy, 48% reduction in false positives, and 72% faster 
investigation turnaround. The project management model introduced in this article outlines the lifecycle for planning, 
building, validating, deploying, and governing these systems, emphasiz 
--


### Strategy 3: Regulatory Policy — recursive with heading preservation

In [14]:
def chunk_regulatory(text: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " "]
    )
    docs = []
    for i, chunk in enumerate(splitter.split_text(text)):
        docs.append(Document(
            page_content=chunk,
            metadata={"doc_type": "regulatory", "chunk_id": i},
        ))
    return docs

# Usage on regulatory PDF
regulatory_docs = chunk_regulatory(raw_texts["regulatory"])
print(f"Regulatory chunks: {len(regulatory_docs)}")
print(regulatory_docs[1].page_content[:411], "\n--")

Regulatory chunks: 53
(FPC). However, despite these guidelines, rising consumer complaints indicate a gap between regulatory 
expectations and actual practices. This study aims to empirically examine the compliance of FPC norms among 
Banks and Housing Finance Companies (HFCs), as perceived by lending officials and borrowers. Primary data 
were collected from 294 borrowers and 102 lending branches using structured questionnaires. 
--


### Strategy 4: Scorecard Cut-off — table-aware chunking

In [15]:
import re

def chunk_scorecard(text: str) -> List[Document]:
    """Detect numeric ranges like 700-750 or 700 - 750 and split accordingly."""
    pattern = re.compile(r"(\d{3})\s*[--to]+\s*(\d{3})")
    matches = list(pattern.finditer(text))

    docs: List[Document] = []
    if not matches:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=700, 
            chunk_overlap=80,
            separators=["\n\n", "\n", ". ", " "]
        )
        for i, chunk in enumerate(splitter.split_text(text)):
            docs.append(Document(
                page_content=chunk,
                metadata={"doc_type": "scorecard", "chunk_id": i, "min_score": None, "max_score": None}
            ))
        return docs

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk = text[start:end].strip()

        if len(chunk) < 30:
            continue
        docs.append(Document(
            page_content=chunk,
            metadata={
                "doc_type": "scorecard",
                "chunk_id": i,
                "min_score": int(m.group(1)),
                "max_score": int(m.group(2))
            }
        ))
    return docs

# Usage on scorecard PDF
score_docs = chunk_scorecard(raw_texts["scorecard"])
print(f"Scorecard chunks: {len(score_docs)}")
print(score_docs[1].page_content[:300], "\n--")

Scorecard chunks: 12
300 to 850.  
 
Credit bureau scores consider five general groups of predictive variables: 
– Previous performance, including the severity and frequency of poor performance and 
how recently the poor performance occurred. 
– Current level and use of nonmortgage debt. 
– Amount of time that credit ha 
--


### Combine all chunks

In [16]:
all_docs: List[Document] = deling_docs + fraud_docs + regulatory_docs + score_docs
print(f"Total chunks: {len(all_docs)}")
print(f"By type: {pd.Series([d.metadata['doc_type'] for d in all_docs]).value_counts().to_dict()}")

Total chunks: 105
By type: {'regulatory': 53, 'fraud': 36, 'scorecard': 12, 'delinquency': 4}


## 🧩 Customer-Row → Narrative Document